# Bayesian inference for phase SDEs (CPU)

This test generates synthetic data by adding noise to the known phase equations of the Gray–Scott model, instead of using simulation data from the Gray–Scott model itself. It then applies Bayesian inference to recover the phase equations and noise covariance for both systems.

Reference data: `ReferenceData/PhaseEquation/`. Estimator: `bayes_cpu.py`.

In [10]:
from pathlib import Path
import sys
import time

import numpy as np
from tqdm.notebook import tqdm

# Run from the repository root or SampleCode/.
SAMPLE_DIR = Path.cwd()
if SAMPLE_DIR.name != 'SampleCode':
    SAMPLE_DIR = SAMPLE_DIR / 'SampleCode'
sys.path.insert(0, str(SAMPLE_DIR.resolve()))
from bayes_cpu import fit, evaluate_nonconstant

REFERENCE_DIR = SAMPLE_DIR.parent / 'ReferenceData' / 'PhaseEquation'
np.set_printoptions(formatter={'float_kind': lambda x: f'{x:.3e}'})

## 1. Model and parameters

$$
d\chi_i=[(c,\omega)^T+\epsilon\Gamma(\chi_i-\chi_j)]\,dt+B\,dW_i,
\qquad BB^T=E=\sigma^2C.
$$

Independent $W_i$ for each system. $C$ is from `SRC/Plot_Estimation_Result.py`.
Use the paper's Fourier orders (`Ms=20, Mt=5`) and 40×40 cells with 250 samples per cell.

In [11]:
with np.load(REFERENCE_DIR / 'CouplingFunction_8193.npz') as source:
    phi_axis = source['delta_phi']
    theta_axis = source['delta_theta']
    Gamma = np.stack((source['Gamma_s'], source['Gamma_t']), axis=-1)
with np.load(REFERENCE_DIR / 'freq_8193.npz') as source:
    c = float(source['c'])
    omega = float(source['omega'])
L = (phi_axis[-1] - phi_axis[0]) / 2


# Noise variance
sigma_squared = 1.0e-10

# E is used only for generation and comparison.
epsilon = 1.0e-6
C = np.array([
    [4.97976032004664812e04, -8.73988532241410212e03],
    [-8.73988532241410212e03, 1.79108047214241105e03],
])
E = sigma_squared * C

Ms, Mt = 20, 5
N_CELLS_PHI, N_CELLS_THETA = 40, 40
POINTS_PER_CELL = 250
INITIALS_PER_CELL_AXIS = 20  # 20x20 initial differences per cell
N_PHI = N_CELLS_PHI * INITIALS_PER_CELL_AXIS
N_THETA = N_CELLS_THETA * INITIALS_PER_CELL_AXIS
DT = 5.0                   # Integration step
DELTA_T = 50.0              # Observation interval
N_INTERVALS = 1             # One interval per trajectory
SEED = 415643
MAX_ITERATIONS = 50
RTOL = 1.0e-5

print(f'c = {c:.3e}, omega = {omega:.3e}, L = {L:.3e}')
print(f'epsilon = {epsilon:.3e}, sigma^2 = {sigma_squared:.3e}')
print(f'Ms = {Ms}, Mt = {Mt}, Fourier modes = {(2*Ms+1)*(2*Mt+1)}')
print(f'Initial conditions = {N_PHI*N_THETA:,}, Candidate samples/system = {N_PHI*N_THETA*N_INTERVALS:,}')
print(f'Cells = {N_CELLS_PHI}×{N_CELLS_THETA}, {POINTS_PER_CELL} samples/cell, '
      f'Samples/system = {N_CELLS_PHI*N_CELLS_THETA*POINTS_PER_CELL:,}')
print(f'Trajectory T = {N_INTERVALS*DELTA_T:.3e}, epsilon*T = {epsilon*N_INTERVALS*DELTA_T:.3e}')
print('Supplied E = sigma_squared * C:')
print(E)

c = 4.313e-02, omega = 2.790e-02, L = 1.250e+02
epsilon = 1.000e-06, sigma^2 = 1.000e-10
Ms = 20, Mt = 5, Fourier modes = 451
Initial conditions = 640,000, Candidate samples/system = 640,000
Cells = 40×40, 250 samples/cell, Samples/system = 400,000
Trajectory T = 5.000e+01, epsilon*T = 5.000e-05
Supplied E = sigma_squared * C:
[[4.980e-06 -8.740e-07]
 [-8.740e-07 1.791e-07]]


## 2. Generate data

Periodic bilinear interpolation and Euler–Maruyama; unwrapped phases.
Use 20×20 initial differences per cell, with one interval ($T=50$) per trajectory.

In [12]:
def coupling(delta):
    """Periodic bilinear interpolation of Gamma(delta_phi, delta_theta)."""
    x = (delta[..., 0] % (2*L)) / (phi_axis[1] - phi_axis[0])
    y = (delta[..., 1] % (2*np.pi)) / (theta_axis[1] - theta_axis[0])
    # Include coordinates rounded to the periodic endpoint.
    ix = np.minimum(x.astype(int), len(phi_axis)-2)
    iy = np.minimum(y.astype(int), len(theta_axis)-2)
    ax, ay = (x - ix)[..., None], (y - iy)[..., None]
    return ((1-ay) * ((1-ax)*Gamma[iy, ix] + ax*Gamma[iy, ix+1])
            + ay * ((1-ax)*Gamma[iy+1, ix] + ax*Gamma[iy+1, ix+1]))

initial_phi, initial_theta = np.meshgrid(
    -L + (np.arange(N_PHI) + 0.5)*(2*L/N_PHI),
    -np.pi + (np.arange(N_THETA) + 0.5)*(2*np.pi/N_THETA), indexing='xy',
)
# Shape: (trajectory, observation, system, [Phi, Theta])
phases = np.zeros((N_PHI*N_THETA, N_INTERVALS+1, 2, 2))
phases[:, 0, 0, 0] = initial_phi.ravel()
phases[:, 0, 0, 1] = initial_theta.ravel()
state = phases[:, 0].copy()
noise_factor = np.linalg.cholesky(E)
rng = np.random.default_rng(SEED)
steps_per_observation = int(round(DELTA_T / DT))
started = time.perf_counter()

for step in tqdm(range(1, N_INTERVALS*steps_per_observation + 1),
                 desc='SDE generation', unit='step'):
    delta = state[:, 0] - state[:, 1]
    drift = np.array([c, omega]) + epsilon*np.stack(
        (coupling(delta), coupling(-delta)), axis=1,
    )
    # Independent system noise; increment covariance DT*E.
    noise = rng.standard_normal(state.shape) @ noise_factor.T
    state += DT*drift + np.sqrt(DT)*noise
    if step % steps_per_observation == 0:
        phases[:, step // steps_per_observation] = state

print(f'SDE generation time = {time.perf_counter()-started:.3e} s')
print(f'Phase array shape = {phases.shape}')

SDE generation:   0%|          | 0/10 [00:00<?, ?step/s]

SDE generation time = 1.564e+00 s
Phase array shape = (640000, 2, 2, 2)


## 3. Select samples

Compute midpoints and velocities within each trajectory.
Select 250 samples per cell without replacement, using the same indices for both systems.

In [13]:
midpoint = (0.5*(phases[:, 1:] + phases[:, :-1])).reshape(-1, 2, 2)
velocity = (np.diff(phases, axis=1) / DELTA_T).reshape(-1, 2, 2)
delta_midpoint = midpoint[:, 0] - midpoint[:, 1]

# Select 250 midpoint samples per periodic cell.
cell_coordinates = ((delta_midpoint + [L, np.pi]) % [2*L, 2*np.pi]
                    / [2*L, 2*np.pi] * [N_CELLS_PHI, N_CELLS_THETA]).astype(int)
cell_ids = cell_coordinates[:, 0] + N_CELLS_PHI*cell_coordinates[:, 1]
selected = []
for cell_id in range(N_CELLS_PHI*N_CELLS_THETA):
    members = np.flatnonzero(cell_ids == cell_id)
    selected.append(rng.choice(members, POINTS_PER_CELL, replace=False))
selected = np.concatenate(selected)
midpoint = midpoint[selected]
velocity = velocity[selected]
delta_midpoint = delta_midpoint[selected]
print(f'Midpoint samples/system = {len(velocity):,}')

Midpoint samples/system = 400,000


## 4. Bayesian inference

Estimate coefficients and unknown $E$ with a fixed prior.
Update order: residuals → $E$ → $\Xi$ → $r$ → $a$, following `SRC/Module_Bayes_GPU.py`.
The supplied $E$ is used only for generation and comparison.

In [14]:
results = []
for system in range(2):
    started = time.perf_counter()
    sign = 1 if system == 0 else -1
    result = fit(
        sign*delta_midpoint, velocity[:, system], DELTA_T, L,
        Ms=Ms, Mt=Mt, max_iterations=MAX_ITERATIONS, rtol=RTOL,
    )
    results.append(result)
    print(f'System {system+1}: estimation time={time.perf_counter()-started:.3e} s', flush=True)

System 1: estimation time=5.769e+00 s
System 2: estimation time=5.862e+00 s


## 5. Frequencies and constant modes

$$
a^s_{0}=c+\epsilon\langle\Gamma_s\rangle,\qquad
a^t_{0}=\omega+\epsilon\langle\Gamma_t\rangle.
$$

Frequencies and coupling means are not separately identifiable.
For comparison, subtract the theoretical coupling means from the estimated constant modes.

In [15]:
gamma_mean = Gamma[:-1, :-1].mean(axis=(0, 1))
frequency_theory = np.array([c, omega])
constant_theory = frequency_theory + epsilon*gamma_mean
zero = np.all(results[0]['modes'] == 0, axis=1)
constant_estimates = np.stack([result['a'][:, zero].real.ravel() for result in results])
frequency_estimates = constant_estimates - epsilon*gamma_mean

print('Theoretical mean Gamma [s, t] =', gamma_mean)
print('\nFourier constant modes (including coupling means):')
for k, name in enumerate(('a_s00', 'a_t00')):
    print(f'{name}: system 1={constant_estimates[0,k]:.3e}, '
          f'system 2={constant_estimates[1,k]:.3e}, theory={constant_theory[k]:.3e}')
print('\nFrequencies after subtracting the theoretical coupling means:')
for k, name in enumerate(('c', 'omega')):
    print(f'{name}: system 1={frequency_estimates[0,k]:.3e}, '
          f'system 2={frequency_estimates[1,k]:.3e}, theory={frequency_theory[k]:.3e}')

Theoretical mean Gamma [s, t] = [-8.820e+01 1.305e+01]

Fourier constant modes (including coupling means):
a_s00: system 1=4.304e-02, system 2=4.304e-02, theory=4.304e-02
a_t00: system 1=2.791e-02, system 2=2.791e-02, theory=2.791e-02

Frequencies after subtracting the theoretical coupling means:
c: system 1=4.313e-02, system 2=4.313e-02, theory=4.313e-02
omega: system 1=2.790e-02, system 2=2.790e-02, theory=2.790e-02


## 6. Fourier coefficients

Magnitudes of the seven modes in [Fig. 10](https://arxiv.org/html/2604.23727#S3.F10). Coefficients include $\epsilon$.
Reference coefficients: `ReferenceData/PhaseEquation/coef_*_{real,imag}.csv`.

In [16]:
shown_modes = np.array([(1, 0), (2, 0), (3, 0), (0, 1), (1, 1), (2, 1), (3, 1)])
shown_indices = [np.flatnonzero(np.all(results[0]['modes'] == mode, axis=1)).item()
                 for mode in shown_modes]
# Stored indices: index_n = ms, index_m = mt; coefficients exclude epsilon.
reference_ms = np.loadtxt(REFERENCE_DIR / 'index_n.csv', delimiter=',', dtype=int)
reference_mt = np.loadtxt(REFERENCE_DIR / 'index_m.csv', delimiter=',', dtype=int)
true_coefficients = []
for component in ('s', 't'):
    coefficients = (np.loadtxt(REFERENCE_DIR / f'coef_{component}_real.csv', delimiter=',')
                    + 1j*np.loadtxt(REFERENCE_DIR / f'coef_{component}_imag.csv', delimiter=','))
    true_coefficients.append(epsilon*np.array([
        coefficients[(reference_ms == ms) & (reference_mt == mt)].item()
        for ms, mt in shown_modes
    ]))
true_coefficients = np.asarray(true_coefficients)
posterior_coefficients = np.stack([result['a'][:, shown_indices] for result in results])


for k, component in enumerate(('s', 't')):
    print(f'\n|a^{component}_m|')
    print(f'{"(ms,mt)":>8} {"Theory":>12} {"System 1":>12} {"System 2":>12}')
    for j, (ms, mt) in enumerate(shown_modes):
        mode = f'({ms},{mt})'
        print(f'{mode:>8} {abs(true_coefficients[k,j]):12.3e} '
              f'{abs(posterior_coefficients[0,k,j]):12.3e} '
              f'{abs(posterior_coefficients[1,k,j]):12.3e}')


|a^s_m|
 (ms,mt)       Theory     System 1     System 2
   (1,0)    4.148e-05    4.186e-05    4.142e-05
   (2,0)    2.603e-05    2.622e-05    2.598e-05
   (3,0)    1.030e-05    9.794e-06    1.012e-05
   (0,1)    3.010e-06    3.206e-06    3.129e-06
   (1,1)    1.891e-06    1.613e-06    2.040e-06
   (2,1)    8.343e-07    5.534e-07    8.183e-07
   (3,1)    2.482e-07    4.272e-07    5.601e-07

|a^t_m|
 (ms,mt)       Theory     System 1     System 2
   (1,0)    8.224e-06    8.324e-06    8.183e-06
   (2,0)    5.555e-06    5.574e-06    5.538e-06
   (3,0)    2.166e-06    2.088e-06    2.133e-06
   (0,1)    2.051e-07    2.129e-07    1.977e-07
   (1,1)    6.970e-08    5.304e-08    8.426e-08
   (2,1)    4.069e-08    4.452e-08    4.347e-08
   (3,1)    6.107e-08    4.639e-08    1.415e-07


## 7. Absolute RMS error

$\hat\Gamma=\Gamma-\langle\Gamma\rangle$.

[Eqs. (71)–(72)](https://arxiv.org/html/2604.23727#S3.SS3), evaluated on the original 2048×2048 periodic grid without duplicate endpoints:

$$
\mathrm{RMS}_p=\sqrt{\frac{1}{4\pi L}\int_{-L}^{L}\!d\Delta\Phi
\int_{-\pi}^{\pi}\!d\Delta\Theta\,
(\epsilon\hat\Gamma^p_{\mathrm{est}}-\epsilon\hat\Gamma^p_{\mathrm{true}})^2},
\qquad p=s,t.
$$

In [17]:
coupling_theory = epsilon*(Gamma[:-1, :-1] - gamma_mean)
paper_errors = []
for system, result in enumerate(results, start=1):
    estimated = evaluate_nonconstant(result, phi_axis[:-1], theta_axis[:-1], L)
    error = np.sqrt(np.mean((estimated-coupling_theory)**2, axis=(0, 1)))
    paper_errors.append(error)
    print(f'System {system}: error[epsilon*Gamma_s_hat] = {error[0]:.3e}')
    print(f'System {system}: error[epsilon*Gamma_t_hat] = {error[1]:.3e}')
del estimated
paper_errors = np.asarray(paper_errors)

System 1: error[epsilon*Gamma_s_hat] = 1.072e-05
System 1: error[epsilon*Gamma_t_hat] = 2.049e-06
System 2: error[epsilon*Gamma_s_hat] = 1.119e-05
System 2: error[epsilon*Gamma_t_hat] = 2.169e-06


## 8. Noise covariance

Rows and columns: $(\Phi,\Theta)$. Compare estimated $E$ with supplied $\sigma^2C$.

In [19]:
print('Supplied E = sigma_squared * C:')
print(E)
for system, result in enumerate(results, start=1):
    print(f'\nSystem {system}: estimated E:')
    print(result['E'])

Supplied E = sigma_squared * C:
[[4.980e-06 -8.740e-07]
 [-8.740e-07 1.791e-07]]

System 1: estimated E:
[[4.974e-06 -8.731e-07]
 [-8.731e-07 1.789e-07]]

System 2: estimated E:
[[4.963e-06 -8.698e-07]
 [-8.698e-07 1.781e-07]]
